In [1]:
print("Here begins the BN models notebook.")

Here begins the BN models notebook.


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/world_happiness_bn_clean.csv")

df_bn.head()

,Country,Climate_Index,Corruption_Perception,Crime_Rate,Education_Index,Employment_Rate,Freedom,GDP_per_Capita,Generosity,Happiness_Score,Healthy_Life_Expectancy,Income_Inequality,Internet_Access,Life_Satisfaction,Mental_Health_Index,Political_Stability,Population,Public_Health_Expenditure,Public_Trust,Social_Support,Unemployment_Rate,Urbanization_Rate,Work_Life_Balance,Year_bin
0,Australia,bin_2,bin_3,bin_2,bin_4,bin_1,bin_0,bin_1,bin_4,bin_2,bin_2,bin_3,bin_4,bin_0,bin_4,bin_3,bin_1,bin_0,bin_2,bin_3,bin_2,bin_2,bin_3,2005
1,Australia,bin_4,bin_1,bin_3,bin_1,bin_2,bin_2,bin_1,bin_0,bin_4,bin_0,bin_0,bin_2,bin_1,bin_4,bin_3,bin_2,bin_1,bin_1,bin_4,bin_0,bin_3,bin_2,2005
2,Australia,bin_3,bin_1,bin_3,bin_4,bin_2,bin_3,bin_4,bin_3,bin_3,bin_1,bin_1,bin_3,bin_4,bin_3,bin_4,bin_1,bin_2,bin_3,bin_2,bin_0,bin_4,bin_2,2005
3,Australia,bin_0,bin_1,bin_3,bin_0,bin_0,bin_4,bin_1,bin_3,bin_4,bin_3,bin_0,bin_4,bin_3,bin_0,bin_4,bin_3,bin_3,bin_3,bin_1,bin_4,bin_4,bin_0,2005
4,Australia,bin_0,bin_3,bin_0,bin_2,bin_2,bin_4,bin_2,bin_0,bin_1,bin_3,bin_4,bin_3,bin_2,bin_0,bin_2,bin_0,bin_3,bin_0,bin_3,bin_3,bin_2,bin_3,2005


In [3]:
# Inspect shape, data types, and missing values
print(df_bn.shape)
print(df_bn.dtypes)

print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(200, 24)
Country                        str
Climate_Index                  str
Corruption_Perception          str
Crime_Rate                     str
Education_Index                str
Employment_Rate                str
Freedom                        str
GDP_per_Capita                 str
Generosity                     str
Happiness_Score                str
Healthy_Life_Expectancy        str
Income_Inequality              str
Internet_Access                str
Life_Satisfaction              str
Mental_Health_Index            str
Political_Stability            str
Population                     str
Public_Health_Expenditure      str
Public_Trust                   str
Social_Support                 str
Unemployment_Rate              str
Urbanization_Rate              str
Work_Life_Balance              str
Year_bin                     int64
dtype: object

Missing values per column:
Country                      0
Climate_Index                0
Corruption_Perception        0
Crime_Rate     

In [4]:
# Convert all columns to string so pgmpy treats them as discrete states
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Country                      string
Climate_Index                string
Corruption_Perception        string
Crime_Rate                   string
Education_Index              string
Employment_Rate              string
Freedom                      string
GDP_per_Capita               string
Generosity                   string
Happiness_Score              string
Healthy_Life_Expectancy      string
Income_Inequality            string
Internet_Access              string
Life_Satisfaction            string
Mental_Health_Index          string
Political_Stability          string
Population                   string
Public_Health_Expenditure    string
Public_Trust                 string
Social_Support               string
Unemployment_Rate            string
Urbanization_Rate            string
Work_Life_Balance            string
Year_bin                     string
dtype: object


In [5]:
# Check number of states per variable
cardinality = pd.DataFrame({
    "Variable": df_bn.columns,
    "Num_States": [df_bn[col].nunique(dropna=True) for col in df_bn.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
0,Country,10
1,Climate_Index,5
2,Corruption_Perception,5
3,Crime_Rate,5
4,Education_Index,5
5,Employment_Rate,5
6,Freedom,5
7,GDP_per_Capita,5
8,Generosity,5
9,Happiness_Score,5


In [6]:
# Create BN modeling dataset
df_bn_model = df_bn.copy()

# Drop Country to avoid high-cardinality country identity dominating the graph
df_bn_model = df_bn_model.drop(columns=["Country"], errors="ignore")

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(200, 23)
['Climate_Index', 'Corruption_Perception', 'Crime_Rate', 'Education_Index', 'Employment_Rate', 'Freedom', 'GDP_per_Capita', 'Generosity', 'Happiness_Score', 'Healthy_Life_Expectancy', 'Income_Inequality', 'Internet_Access', 'Life_Satisfaction', 'Mental_Health_Index', 'Political_Stability', 'Population', 'Public_Health_Expenditure', 'Public_Trust', 'Social_Support', 'Unemployment_Rate', 'Urbanization_Rate', 'Work_Life_Balance', 'Year_bin']
